In [ ]:
import statsapi
import itertools
import pandas as pd
import time
import os
from pathlib import Path

In [ ]:
def get_players_api(season):
    all_players = []

    teams = statsapi.get("teams", {"sportIds": 1, "season": season})['teams']
    team_ids = [team['id'] for team in teams]

    for tid in team_ids:
        roster = statsapi.get('team_roster', {'teamId': tid, 'season': season})['roster']
        for player in roster:
            # OPTIONAL: Skip pitchers to save 50% of API calls and go 2x faster
            if player.get('position', {}).get('abbreviation') == 'P':
                continue
                
            id = player['person']['id']
            all_players.append(id)
        time.sleep(0.1)

    all_players = sorted(list(set(all_players))) 
    print(f"Total Hitters to collect for {season}: {len(all_players)}")
    
    all_data = []
    for index, id in enumerate(all_players):
        max_retries = 3
        for attempt in range(max_retries):
            try:
                stat = statsapi.player_stat_data(id, group="hitting", type="season", season=season)
                if not stat.get('stats'):
                    break
                
                keys_to_remove = ["active", "nickname", "last_played"]
                for k in keys_to_remove: stat.pop(k, None)
                
                stat_season = dict(itertools.islice(stat["stats"][0].items(), 0, 3))
                real_stat = stat["stats"][0]["stats"]
                stat_player = dict(itertools.islice(stat.items(), 0, 8))
                
                all_data.append({**stat_player, **stat_season, **real_stat})
                print(f"Processed Index: {index}/{len(all_players)}")
                break 
                
            except Exception as e:
                print(f"Error {id}: {e}. Retrying in 10s...")
                time.sleep(10)

        time.sleep(0.4) # Normal delay

    df = pd.DataFrame(all_data)
    
    # Ensure folder exists so it doesn't crash at the very end
    output_dir = Path.cwd() / "data" / "processed" / "left_right" / "data_attack"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{season}.csv"
    df.to_csv(output_path, index=False)
    print(f"Success! Saved to {output_path}")


In [ ]:
for season in range(2011, 2005, -1):
    get_players_api(season=season)